In [14]:
# SUPPORT IMPORTS
import os
import warnings

# DATA ANALYSIS IMPORTS
import numpy as np
import pandas as pd

# DATASET LOADING IMPORTS
import torch
from datasets import load_dataset
import evaluate

# MODEL IMPORTS
from transformers import AutoTokenizer, AutoModelForTokenClassification
from transformers import DataCollatorForTokenClassification
from transformers import TrainingArguments, Trainer

In [15]:
# SETTINGS
warnings.filterwarnings('ignore')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"USING DEVICE: {device}")

USING DEVICE: cpu


In [16]:
# CONSTANT DECLARATIONS
MODEL_CHECKPOINT = "distilbert-base-uncased"
BATCH_SIZE = 16
LEARNING_RATE = 2e-5
NUM_EPOCHS = 2
WEIGHT_DECAY = 0.01
OUTPUT_DIR = "./ner_results"
SAVE_DIR = "./my_baseline_ner"

In [17]:
# LOAD CONLL2003 DATASET
raw_datasets = load_dataset("conll2003")

# EXTRACT LABEL LIST
label_list = raw_datasets["train"].features["ner_tags"].feature.names
print("LABELS FOUND:", label_list)

LABELS FOUND: ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']


In [18]:
TOKENIZER = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

def TOKEN_IT(ITEM):
    TOKENIZED = TOKENIZER(ITEM["tokens"], truncation=True, is_split_into_words=True)
    labels = []
    for i, label in enumerate(ITEM["ner_tags"]):
        WORD_IDXS = TOKENIZED.word_ids(batch_index=i)
        PREVIOUS_IDX = None
        LABEL_IDX = []
        for WORD_IDX in WORD_IDXS:
            if WORD_IDX is None:
                LABEL_IDX.append(-100)
            elif WORD_IDX != PREVIOUS_IDX:
                LABEL_IDX.append(label[WORD_IDX])
            else:
                LABEL_IDX.append(-100)
            PREVIOUS_IDX = WORD_IDX
        labels.append(LABEL_IDX)
    TOKENIZED["labels"] = labels
    return TOKENIZED

tokenized_datasets = raw_datasets.map(TOKEN_IT, batched=True)

In [19]:
# LOAD SEQEVAL METRIC
metric = evaluate.load("seqeval")

# DEFINE METRICS FUNCTION
def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)
    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    results = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

In [20]:
id2label = {i: label for i, label in enumerate(label_list)}
label2id = {label: i for i, label in enumerate(label_list)}

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=WEIGHT_DECAY,
)

data_collator = DataCollatorForTokenClassification(TOKENIZER)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
# START TRAINING
trainer.train()

# SAVE FINISHED MODEL
trainer.save_model(SAVE_DIR)
print("MODEL SAVED LOCALLY.")

In [21]:
# LOAD SAVED TOKENIZER
TOKENIZER = AutoTokenizer.from_pretrained(SAVE_DIR)
# LOAD SAVED MODEL
model = AutoModelForTokenClassification.from_pretrained(SAVE_DIR)
print("SAVED MODEL LOADED SUCCESSFULLY.")

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

SAVED MODEL LOADED SUCCESSFULLY.


In [22]:
# NEW TRAINING PARAMETERS
NEW_NUM_EPOCHS = 5
NEW_OUTPUT_DIR = "./PRE-TRAINED"

args = TrainingArguments(
    output_dir=NEW_OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",   
    logging_strategy="epoch",
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=NEW_NUM_EPOCHS,
    weight_decay=WEIGHT_DECAY,
)

In [23]:
# DATA COLLATOR FOR TOKEN CLASSIFICATION
data_collator = DataCollatorForTokenClassification(TOKENIZER)

In [24]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [26]:
# CONTINUE TRAINING FROM SAVED MODEL
trainer.train()

print("CONTINUED TRAINING FINISHED.")

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.036151,0.049256,0.924252,0.936385,0.930279,0.986936
2,0.019210,0.053385,0.924769,0.941266,0.932944,0.986761
3,0.012216,0.054853,0.933289,0.944127,0.938676,0.987500
4,0.008154,0.055927,0.932127,0.945305,0.938670,0.987325
5,0.005714,0.057464,0.934033,0.943622,0.938803,0.987617


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

CONTINUED TRAINING FINISHED.


In [27]:
# SAVE UPDATED MODEL
FINAL_SAVE_DIR = "./my_baseline_ner_continued"

trainer.save_model(FINAL_SAVE_DIR)

print("UPDATED MODEL SAVED LOCALLY.")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

UPDATED MODEL SAVED LOCALLY.


In [28]:
# FINAL EVALUATION
FINAL_METRICS = trainer.evaluate()

print("FINAL METRICS:", FINAL_METRICS)

FINAL METRICS: {'eval_loss': 0.05746355652809143, 'eval_precision': 0.9340329835082459, 'eval_recall': 0.9436216762032985, 'eval_f1': 0.9388028463792382, 'eval_accuracy': 0.9876173046220942, 'eval_runtime': 92.536, 'eval_samples_per_second': 35.121, 'eval_steps_per_second': 2.205, 'epoch': 5.0}
